# Lab 11 — Features & selection with leakage controls

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 11 — §11.3–§11.4 (feature families; features that separate sleep stages), §11.6–§11.7 (scaling and the leakage it hides; selection).

**Biomedical question.** Which features separate the sleep stages — and does my feature / selection / scaling pipeline secretly *leak*?
**Task type (§1.8).** Feature extraction + honest selection (which features generalise to a NEW subject)
**Information that must be preserved.** the *claim*: the reported score must represent an UNSEEN subject, so scaling + selection are fit INSIDE the split and whole subjects are held out
**Main assumptions.** every subject adds an idiosyncratic per-feature offset (a nuisance / leak vector); the class structure itself is shared across subjects
**Primary diagnostic.** the GAP between a pipeline fit on ALL data + a random epoch split (inflated) and a leak-free pipeline (scale+select inside the fold) + a subject-grouped split (honest)
**Transfer challenge.** repeat with an embedded (model-importance) selector, or redesign the split & metrics for a new-DEVICE claim

*Self-contained: one seeded synthetic multi-subject cohort, no `bsp`, no I/O; runs offline in well under a minute.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab11_features_and_leakage_controls/lab11_features_and_leakage_controls.ipynb) [![View](https://img.shields.io/badge/view-static-orange)](https://farhad-abtahi.github.io/CM2013/nb/lab11_features_and_leakage_controls.html) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab11_features_and_leakage_controls.ipynb)

In [ ]:
# --- shared setup (reproducible; fully offline synthetic cohort) ---
import numpy as np, matplotlib.pyplot as plt
from functools import partial
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (cross_val_score, cross_val_predict,
                                     GroupKFold, KFold, GroupShuffleSplit)
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

# A deterministic MI scorer: mutual_info_classif uses a randomised kNN estimator, so we PIN
# its random_state -- otherwise the ranking (and every downstream score) would wobble run to run.
mi_scorer = partial(mutual_info_classif, random_state=0)
K_TOP = 3          # keep the top-K features that MI says separate the stages
CLASSES = ["W", "N2", "N3"]

def make_cohort(n_subj=9, per=90, offset_std=2.5, noise_std=0.9, seed=2013):
    """Synthetic per-epoch FEATURE vectors for a multi-subject sleep cohort.

    8 features. Three (0,1,2) carry real, subject-INVARIANT class structure; feature 3 is
    weakly informative; features 4-7 are pure noise. On top of the class means we add ONE
    per-subject offset vector (the 'leak vector') that shifts *all* of that subject's epochs
    the same way regardless of class. Because the offset spread (>=2.5) dwarfs the class-mean
    gaps (~1-2), the classes OVERLAP heavily once subjects are pooled -- yet inside any single
    subject they are still separable. A model that sees a subject in BOTH train and test can
    exploit that, which is exactly the leak this lab exposes. Returns X, y, groups.
    """
    r = np.random.default_rng(seed)
    M = np.zeros((3, 8))                      # class means, rows = W / N2 / N3
    M[:, 0] = [ 1.3,  0.0, -1.1]              # informative
    M[:, 1] = [-0.9,  1.2,  0.1]              # informative
    M[:, 2] = [ 0.2, -1.0,  1.1]              # informative
    M[:, 3] = [ 0.4,  0.0, -0.4]              # weakly informative
    X, y, g = [], [], []
    for s in range(n_subj):
        offset = r.normal(0, offset_std, size=8)          # subject-specific nuisance (leak vector)
        for i in range(per):
            cls = i % 3
            X.append(M[cls] + offset + r.normal(0, noise_std, size=8))
            y.append(cls); g.append(s)
    return np.array(X), np.array(y), np.array(g)


## 1. Build & inspect the cohort
Make the dataset and *see* the two facts that drive this lab: (a) pooled across subjects the classes **overlap**, and (b) each subject sits at its own **offset** in feature space. The offset is the nuisance a pooled pipeline can secretly exploit.

In [ ]:
X, y, groups = make_cohort()
n_subj = len(np.unique(groups))
print("X", X.shape, " classes", np.bincount(y), " subjects", n_subj)

# Evidence of the offset: per-subject feature means differ far more than the class means do.
subj_means = np.array([X[groups == s].mean(0) for s in range(n_subj)])   # (n_subj, 8)
class_gap  = X[y == 0].mean(0) - X[y == 2].mean(0)                        # W - N3, pooled
print("per-subject offset spread (std of subject means), feats 0-3:", subj_means.std(0)[:4].round(2))
print("pooled class gap |W-N3|, feats 0-3:                        ", np.abs(class_gap)[:4].round(2))
print("-> subject offset >> class gap: pooled classes overlap, but each subject is a tight cluster")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for c in range(3):
    m = y == c; ax[0].scatter(X[m, 0], X[m, 1], s=6, alpha=.5, label=CLASSES[c])
ax[0].set_title("coloured by CLASS (pooled -> overlap)"); ax[0].legend(fontsize=8)
ax[0].set_xlabel("feature 0"); ax[0].set_ylabel("feature 1")
for s in range(n_subj):
    m = groups == s; ax[1].scatter(X[m, 0], X[m, 1], s=6, alpha=.5)
ax[1].set_title("coloured by SUBJECT (offset clusters)"); ax[1].set_xlabel("feature 0")
plt.tight_layout(); plt.show()
# Checkpoint: in the left panel could you draw ONE clean class boundary? Why does the right
# panel warn you that a pooled model might just be memorising which subject a point came from?

## 2. Rank features by mutual information (on TRAINING data only)
`# TODO` split OFF whole subjects for training, then score each feature with `mutual_info_classif` **on the training rows only** and print the ranking. Computing MI on the full dataset would already be a small leak — do it the honest way here.

In [ ]:
# TODO make a subject-grouped train/test split, then rank the 8 features by mutual information
#   computed on the TRAINING rows only (use mi_scorer / mutual_info_classif with random_state=0).
#   Print each feature index with its MI, sorted high->low.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: which features rise to the top, and do they match the ones you DESIGNED to carry
# class structure (0,1,2)? The noise features should sit near MI = 0.

## 3a. The leak — an inflated score
`# TODO` do it the tempting (wrong) way: fit the `StandardScaler` **and** the top-K selector on the **whole** dataset, then score with a **random** epoch split (`KFold(shuffle=True)`). Both moves leak: preprocessing sees the test rows, and the random split scatters each subject across train AND test. Save the number as `acc_inflated`.

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=0)
# TODO fit scaling + top-K selection on ALL of X (leak), then cross_val_score with a random
#   KFold(5, shuffle=True). Store the mean accuracy in acc_inflated and print it.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: name the TWO leaks in the cell above (preprocessing on test rows; subject in both folds).

## 3b. The honest score — pipeline + subject-grouped split
`# TODO` now do it correctly: put scaling **and** selection inside a `Pipeline` so they are refit on each training fold only, and evaluate with `GroupKFold` so **whole subjects** are held out. Store `acc_honest`, then print inflated, honest, and the **GAP**.

In [ ]:
# TODO build Pipeline([StandardScaler, SelectKBest(mi_scorer, k=K_TOP), clf]) and score it with
#   GroupKFold(5, groups=groups). Store the mean in acc_honest, then print inflated / honest / GAP.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: chance is 1/3 ~ 0.33 -- is the HONEST score still above chance? Then the top features
# really do separate stages; the inflated number just over-credited the pipeline for knowing subjects.

## 4. Live sanity check
A leak claim you never verify is untrustworthy. Confirm, from the computed objects, that (i) the gap is positive and (ii) the grouped split never lets one subject sit in train **and** test — while the random split does exactly that.

In [ ]:
# --- sanity check: the leak is real and the grouped split is subject-clean (computed live) ---
assert acc_inflated > acc_honest, "expected the inflated score to beat the honest one"
print(f"[ok] leak gap is positive: {acc_inflated - acc_honest:.3f}")

# grouped folds: train and test subjects must be DISJOINT
for i, (a, b) in enumerate(GroupKFold(5).split(X, y, groups)):
    assert set(groups[a]).isdisjoint(set(groups[b])), f"subject spans fold {i}!"
print(f"[ok] GroupKFold: every fold holds WHOLE subjects out (no subject in train+test)")

# random KFold: at least one subject DOES straddle train and test -- the leak channel
straddle = [s for a, b in KFold(5, shuffle=True, random_state=0).split(X)
            for s in set(groups[a]) & set(groups[b])]
print(f"[ok] random KFold: subjects appearing in BOTH train and test (the leak) = "
      f"{sorted(map(int, set(straddle)))} -> all {n_subj}")

## Reflection
1. Which conclusion stayed **stable** across the two evaluations, and which one **changed**? (Contrast *which features carry class information* — the MI ranking — with *the accuracy you would put in the abstract*.)
2. What evidence would convince a reviewer this pipeline works on a **new population** (new subjects, new site)? Name the split, what is fit inside it, and what you would report alongside the point estimate.
3. Your headline accuracy dropped once you closed the leak. Write the one-sentence caption you would now put under the honest number.

**Rule out.** Fitting the `StandardScaler` / `SelectKBest` on *all* rows before cross-validation, or using an epoch-level (row-shuffled) split that places one subject in both train and test, is ruled out: it lets test information reach the model and breaks the §1.8 requirement that the reported number represent an *unseen subject* — the same leakage failure dissected in Ch 12.

> *Your answers here.*